# Mouth Movement Detection - Getting Started

This notebook demonstrates the complete pipeline for mouth movement detection:
1. Data collection
2. Feature extraction
3. Model training
4. Evaluation
5. Real-time inference

## Setup and Imports

In [ ]:
import sys
import os

# Add src to path
sys.path.append('../src')

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import cv2
from pathlib import Path

# Set style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

print(f"PyTorch version: {torch.__version__}")
print(f"Device: {torch.device('cuda' if torch.cuda.is_available() else 'cpu')}")

## Step 1: Test Facial Landmark Detection

In [ ]:
from features.facial_landmarks import FacialLandmarkDetector

# Initialize detector
detector = FacialLandmarkDetector(detector_type="mediapipe")

# Test on webcam (uncomment to run)
# This will open a window - press 'q' to quit
# !python ../src/features/facial_landmarks.py

## Step 2: Test Feature Extraction

In [ ]:
from features.mouth_features import MouthFeatureExtractor

# Initialize feature extractor
extractor = MouthFeatureExtractor(temporal_window=3)

# Test on webcam (uncomment to run)
# !python ../src/features/mouth_features.py

## Step 3: Collect Training Data

Run the data collection script to record training data.

**Instructions:**
1. The script will open your webcam
2. Press SPACE to start/stop recording
3. Record yourself speaking (mouth moving)
4. Record yourself silent (mouth not moving)
5. Press 's' to save the session
6. Press 'q' to quit

In [ ]:
# Collect data (uncomment to run)
# This will open a webcam window
# !python ../src/data_processing/collect_data.py --output ../data/raw --duration 300

print("After collecting data, label it using:")
print("python ../src/data_processing/label_data.py --session ../data/raw/[SESSION_NAME] --mode manual")
print("Or use auto-labeling:")
print("python ../src/data_processing/label_data.py --session ../data/raw/[SESSION_NAME] --mode auto")

## Step 4: Explore Collected Data

In [ ]:
from data_processing.dataset import load_session_data, load_multiple_sessions

# Load data from all sessions
data_dir = "../data/raw"

try:
    features, labels = load_multiple_sessions(data_dir)
    
    print(f"\nDataset Summary:")
    print(f"Total samples: {len(labels)}")
    print(f"Feature dimension: {features.shape[1]}")
    print(f"Moving samples: {np.sum(labels == 1)} ({np.mean(labels)*100:.1f}%)")
    print(f"Not moving samples: {np.sum(labels == 0)} ({(1-np.mean(labels))*100:.1f}%)")
    
    # Plot class distribution
    plt.figure(figsize=(8, 5))
    plt.bar(['Not Moving', 'Moving'], [np.sum(labels == 0), np.sum(labels == 1)])
    plt.ylabel('Number of Samples')
    plt.title('Class Distribution')
    plt.show()
    
    # Plot feature distributions
    fig, axes = plt.subplots(2, 3, figsize=(15, 8))
    axes = axes.ravel()
    
    feature_names = ['MAR', 'Dist 1', 'Dist 2', 'Temporal 1', 'Intensity Mean', 'Edge Mean']
    feature_indices = [0, 1, 2, 11, 21, 23]
    
    for i, (ax, name, idx) in enumerate(zip(axes, feature_names, feature_indices)):
        moving = features[labels == 1, idx]
        not_moving = features[labels == 0, idx]
        
        ax.hist(not_moving, bins=30, alpha=0.5, label='Not Moving', density=True)
        ax.hist(moving, bins=30, alpha=0.5, label='Moving', density=True)
        ax.set_xlabel(name)
        ax.set_ylabel('Density')
        ax.legend()
    
    plt.tight_layout()
    plt.show()
    
except Exception as e:
    print(f"Error loading data: {e}")
    print("Please collect and label data first!")

## Step 5: Train the Model

In [ ]:
# Train model (uncomment to run)
# This will take several minutes depending on your data size
# !python ../src/models/train.py --config ../config/train_config.yaml

print("Training will:")
print("1. Load all collected data")
print("2. Split into train/val/test sets")
print("3. Train the neural network")
print("4. Save the best model")
print("5. Evaluate on test set")
print("\nMonitor training with: tensorboard --logdir logs")

## Step 6: Evaluate the Trained Model

In [ ]:
from models.network import create_model
from utils.metrics import MetricsCalculator
from data_processing.dataset import create_data_loaders
import joblib

# Load trained model
model_path = "../models/checkpoints/best_model.pth"

if os.path.exists(model_path):
    checkpoint = torch.load(model_path, map_location='cpu')
    
    # Create model
    config = checkpoint['config']['model']
    model = create_model(
        model_type="standard",
        input_dim=config['input_size'],
        hidden_dim=config['hidden_size'],
        dropout=config['dropout']
    )
    
    model.load_state_dict(checkpoint['model_state_dict'])
    model.eval()
    
    print("Model loaded successfully!")
    print(f"Training metrics: {checkpoint.get('metrics', 'N/A')}")
    
    # Load test data
    try:
        features, labels = load_multiple_sessions("../data/raw")
        train_loader, val_loader, test_loader, scaler = create_data_loaders(
            features, labels, batch_size=32
        )
        
        # Evaluate
        metrics_calc = MetricsCalculator()
        
        with torch.no_grad():
            for features_batch, labels_batch in test_loader:
                outputs = model(features_batch)
                predictions = (outputs >= 0.5).float()
                metrics_calc.update(predictions, labels_batch, outputs)
        
        # Print metrics
        metrics_calc.print_metrics("Test Set")
        
        # Plot confusion matrix
        metrics_calc.plot_confusion_matrix()
        plt.show()
        
        # Plot ROC curve
        metrics_calc.plot_roc_curve()
        plt.show()
        
    except Exception as e:
        print(f"Error evaluating: {e}")
else:
    print(f"Model not found at {model_path}")
    print("Please train the model first!")

## Step 7: Test Real-Time Inference

In [ ]:
# Run real-time inference (uncomment to run)
# This will open a webcam window
# !python ../src/inference.py --model ../models/checkpoints/best_model.pth

print("Real-time inference will:")
print("1. Open your webcam")
print("2. Detect your face and mouth")
print("3. Extract features in real-time")
print("4. Classify mouth movement (MOVING / NOT MOVING)")
print("5. Display FPS and inference time")
print("\nPress 'q' to quit")

## Summary

This notebook demonstrated the complete pipeline:

1. ✅ Facial landmark detection
2. ✅ Feature extraction (MAR, distances, temporal, intensity, edges)
3. ✅ Data collection and labeling
4. ✅ Model training with PyTorch
5. ✅ Evaluation metrics (accuracy, precision, recall, F1, ROC)
6. ✅ Real-time inference

### Next Steps:

- Collect more diverse training data (different lighting, angles, speakers)
- Experiment with hyperparameters
- Try the deeper network architecture
- Optimize for specific use cases
- Deploy to production application